In [3]:
from sklearn.model_selection import train_test_split
import pandas as pd


student_df = pd.read_csv('../data/raw/student_mental_health.csv')

X = student_df.drop(columns=["Student_ID", "Mental_Health_Risk", "Risk_Encoded"], errors='ignore')
y = student_df['Mental_Health_Risk']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Shape X_train :", X_train.shape)
print("Shape X_test  :", X_test.shape)

Shape X_train : (1600, 15)
Shape X_test  : (400, 15)


In [4]:
ordinal_cols = ["Physical_Activity", "Diet_Quality"]
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.drop(ordinal_cols).tolist()

print(f"Variables numériques ({len(num_cols)}) :", num_cols)
print(f"Variables catégorielles ({len(cat_cols)}) :", cat_cols)


Variables numériques (10) : ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']
Variables catégorielles (3) : ['Gender', 'Part_Time_Job', 'Previous_Mental_Health_History']


In [5]:
import numpy as np 


for col in num_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    X_train[col] = np.clip(X_train[col], lower_bound, upper_bound)
    X_test[col] = np.clip(X_test[col], lower_bound, upper_bound)

print("Traitement des valeurs aberrantes (IQR Capping) effectué avec succès.")

Traitement des valeurs aberrantes (IQR Capping) effectué avec succès.


In [6]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline



num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
            categories=[
                ['Low', 'Moderate', 'High'],
                ["Poor", "Average", "Good"]
            ],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'KNN':KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1),
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    trained_models[name] = model

In [8]:
from sklearn.metrics import(
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

prediction = {}
results = []

for name,model in trained_models.items():
    y_pred = model.predict()

TypeError: KNeighborsClassifier.predict() missing 1 required positional argument: 'X'